# 1 lines headways-tg

Converted from a Marimo HTML export to a Jupyter Notebook (`.ipynb`).
Code order follows the original Marimo app.


In [ ]:
import pandas as pd
import numpy as np
import gtfs_kit as gk
from datetime import datetime, timedelta, time
import datetime as dt

import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px


In [ ]:
import marimo as mo


### Load time table


In [ ]:
trips = pd.read_csv("data/feed/trips.csv" , low_memory=False)
trips


In [ ]:
timetable = pd.read_csv("data/timetable.csv", low_memory=False)

timetable = timetable.merge(trips[['service_id', 'trip_id' ,'trip_headsign']],
                            how="inner", on = ['service_id', 'trip_id'])

timetable = timetable[['service_id', 'trip_id' , 'date', 'route_short_name',       'trip_headsign', 'arrival_time',  'stop_id', 'stop_name', 'stop_sequence', 
'route_type', 'direction_id','shape_id', 'stop_lat', 'stop_lon', 'gtfs_period']]

timetable['date'] = pd.to_datetime(timetable['date'])
timetable


### headways


In [ ]:
def calculate_headway(big_table, route_short_name, stop_id, trip_headsign=None, date=None):
    """
    Calculate headway (time interval between consecutive departures) for a specific route at a specific stop.

    Parameters:
    -----------
    big_table : pd.DataFrame
        DataFrame containing stop_times merged with route information
        Must include columns: route_short_name, stop_id, service_id, arrival_time, trip_id
    route_short_name : str
        The route identifier (e.g., '87', '1', 'T4')
    stop_id : str
        The stop identifier
    trip_headsign : str, optional
        Filter by trip direction/destination (e.g., 'ERASME', 'GARE DU MIDI')
    date : str, optional
        Filter by specific date (e.g., '20250815')

    Returns:
    --------
    pd.DataFrame
        DataFrame with headway information including:
        - service_id: Service identifier
        - arrival_time: Scheduled arrival time
        - trip_id: Trip identifier
        - headway_minutes: Time difference in minutes from previous trip
        - headway_seconds: Time difference in seconds from previous trip
    """

    # Filter data for the specific route and stop
    filtered_df = big_table[
        (big_table['route_short_name'] == route_short_name) & 
        (big_table['stop_id'] == stop_id)
    ].copy()

    # Apply additional filters if provided
    if trip_headsign is not None:
        filtered_df = filtered_df[filtered_df['trip_headsign'] == trip_headsign]

    if date is not None:
        filtered_df = filtered_df[filtered_df['date'] == date]

    if len(filtered_df) == 0:
        print(f"Warning: No data found for route {route_short_name} at stop {stop_id}")
        if trip_headsign:
            print(f"  with headsign: {trip_headsign}")
        if date:
            print(f"  on date: {date}")
        return pd.DataFrame()

    # Convert arrival_time to seconds since midnight if not already done
    # This handles both time objects and string formats
    if 'arrival_time_seconds' not in filtered_df.columns:
        def time_to_seconds(time_val):
            """Convert time object or string to seconds since midnight"""
            if pd.isna(time_val):
                return None

            # If it's already a time object
            if isinstance(time_val, dt.time):
                return time_val.hour * 3600 + time_val.minute * 60 + time_val.second

            # If it's a string (GTFS format "HH:MM:SS")
            if isinstance(time_val, str):
                parts = time_val.split(':')
                hours = int(parts[0])
                minutes = int(parts[1])
                seconds = int(parts[2])
                return hours * 3600 + minutes * 60 + seconds

            return None

        # Use arrival_time_obj if exists, otherwise parse arrival_time string
        if 'arrival_time_obj' in filtered_df.columns:
            filtered_df['arrival_time_seconds'] = filtered_df['arrival_time_obj'].apply(time_to_seconds)
        else:
            filtered_df['arrival_time_seconds'] = filtered_df['arrival_time'].apply(time_to_seconds)

    # Sort by service_id and arrival_time_seconds
    filtered_df = filtered_df.sort_values(by=['service_id', 'arrival_time_seconds'])

    # Calculate headway within each service_id group
    # Headway = time difference between consecutive trips (in seconds)
    filtered_df['headway_seconds'] = filtered_df.groupby('service_id')['arrival_time_seconds'].diff()

    # Convert to minutes for easier interpretation
    filtered_df['headway_minutes'] = filtered_df['headway_seconds'] / 60

    return filtered_df


In [ ]:
def calculate_headway_by_line(big_table, route_short_name, trip_headsign=None, date=None):
    """
    Calculate headway for all stops along a specific route line.

    This function identifies all stops in sequence for the given route and 
    calculates headway at each stop using the calculate_headway function.

    Parameters:
    -----------
    big_table : pd.DataFrame
        DataFrame containing stop_times merged with route information
        Must include columns: route_short_name, stop_id, stop_sequence, service_id, arrival_time
    route_short_name : str
        The route identifier (e.g., '87', '1', 'T4')
    trip_headsign : str, optional
        Filter by trip direction/destination (e.g., 'ERASME', 'GARE DU MIDI')
    date : str, optional
        Filter by specific date (e.g., '20250815')

    Returns:
    --------
    pd.DataFrame
        DataFrame with headway information for all stops along the route
    """

    # Filter data for the specific route
    route_data = big_table[
        big_table['route_short_name'] == route_short_name
    ].copy()

    # Apply additional filters if provided
    if trip_headsign is not None:
        route_data = route_data[route_data['trip_headsign'] == trip_headsign]

    if date is not None:
        route_data = route_data[route_data['date'] == date]

    if len(route_data) == 0:
        print(f"Warning: No data found for route {route_short_name}")
        if trip_headsign:
            print(f"  with headsign: {trip_headsign}")
        if date:
            print(f"  on date: {date}")
        return pd.DataFrame()

    stops_on_route = route_data.copy()
    numbers_of_stops = stops_on_route['stop_id'].nunique()

    print(f"Found {numbers_of_stops} stops for route {route_short_name}")
    if trip_headsign:
        print(f"  Direction: {trip_headsign}")
    if date:
        print(f"  Date: {date}")

    # Initialize list to collect results
    all_headways = []

    # Iterate through each stop in sequence
    for stop_id in stops_on_route['stop_id'].unique():
        stop_row = stops_on_route[stops_on_route['stop_id'] == stop_id].iloc[0]
        stop_seq = stop_row['stop_sequence']
        stop_name = stop_row.get('stop_name', 'Unknown')

        print(f"  Processing stop {stop_seq}: {stop_name} (ID: {stop_id})")

        # Calculate headway for this specific stop with filters
        headway_data = calculate_headway(stops_on_route, route_short_name, stop_id, 
                                         trip_headsign=trip_headsign, date=date)

        if len(headway_data) > 0:
            # Add stop_sequence to the result
            headway_data['stop_sequence'] = stop_seq

            # Add stop_name if not already present
            if 'stop_name' not in headway_data.columns:
                headway_data['stop_name'] = stop_name

            all_headways.append(headway_data)
        else:
            print(f"    Warning: No headway data calculated for stop {stop_id}")

    # Combine all results
    if len(all_headways) > 0:
        result_df = pd.concat(all_headways, ignore_index=True)

        # Reorder columns for better readability
        column_order = ['stop_sequence', 'stop_id', 'stop_name', 'service_id', 
                       'arrival_time', 'trip_id', 'headway_minutes', 'headway_seconds']

        # Add any additional columns that exist
        for col in result_df.columns:
            if col not in column_order:
                column_order.append(col)

        # Select only columns that exist in the dataframe
        column_order = [col for col in column_order if col in result_df.columns]
        result_df = result_df[column_order]

        print(f"\n✓ Successfully calculated headways for {numbers_of_stops} stops")
        print(f"  Total records: {len(result_df)}")

        return result_df.sort_values(['date', 'stop_id', 'arrival_time_obj']).reset_index(drop=True)
    else:
        print(f"Warning: No headway data could be calculated for route {route_short_name}")
        return pd.DataFrame()


In [ ]:
def calculate_all_headways(big_table):
    """
    Calcule les headways pour toutes les lignes, directions et arrêts.

    Parameters
    ----------
    big_table : pd.DataFrame
        Table fusionnée issue du GTFS (comme dans ton test.py)
    date : str, optional
        Filtre sur une date spécifique (ex: '20250815')

    Returns
    -------
    pd.DataFrame
        Résumé des headways pour toutes les lignes, directions et stops
    """
    all_results = []

    # Liste de toutes les lignes
    all_routes = big_table['route_short_name'].unique()
    print(f"📊 Nombre total de lignes : {len(all_routes)}")

    for route in tqdm(all_routes, desc='Line hw ...'):
        route_data = big_table[big_table['route_short_name'] == route]
        all_headsigns = route_data['trip_headsign'].unique()

        print(f"\n🚍 Ligne {route} — {len(all_headsigns)} directions trouvées")

        for headsign in all_headsigns:
            for date in big_table[(big_table['route_short_name'] == route) & 
                (big_table['trip_headsign'] == headsign)]['date'].unique():
                try:
                    df_line = calculate_headway_by_line(
                        big_table,
                        route_short_name=route,
                        trip_headsign=headsign,
                        date=date
                    )
                    if not df_line.empty:
                        df_line['route_short_name'] = route
                        df_line['trip_headsign'] = headsign
                        all_results.append(df_line)
                except Exception as e:
                    print(f"⚠️ Erreur sur ligne {route}, direction {headsign} : {e}")

    if len(all_results) > 0:
        final_df = pd.concat(all_results, ignore_index=True)
        print(f"\n✅ Headways calculés pour {len(final_df)} enregistrements au total.")
        return final_df
    else:
        print("\n❌ Aucun headway n’a pu être calculé.")
        return pd.DataFrame()


### all headways


In [ ]:
def convert_to_time(time_str):
    """Convert HH:MM:SS string to time object, with 24:00:00 = midnight"""
    if pd.isna(time_str):
        return None

    parts = time_str.split(':')
    hour = int(parts[0])
    minute = int(parts[1])
    second = int(parts[2])

    # Convert 24h to 0h (midnight)
    if hour >= 24:
        hour = hour % 24
    # print(hour, minute, second)
    return time(hour, minute, second)


def extract_hour(time_str):
    """Convert HH:MM:SS string to hour (0-23), with 24:00:00 = 0"""
    if pd.isna(time_str):
        return None

    parts = time_str.split(':')
    hour = int(parts[0])

    # Convert 24h to 0h (midnight)
    # if hour >= 24:
    #     hour = hour % 24
    # print(hour, minute, second)
    return hour


In [ ]:
timetable ['arrival_time_obj'] = timetable['arrival_time'].apply(convert_to_time)
timetable ['arrival_hour'] = timetable['arrival_time'].apply(extract_hour)


In [ ]:
timetable


In [ ]:
all_headways[(all_headways.route_short_name == '71') & ( all_headways.trip_headsign == 'DE BROUCKERE') ] [['stop_id','stop_name']].value_counts()


### save or load all headways


In [ ]:
all_headways.to_parquet("data/stib_all_headways.parquet", index=False)


In [ ]:
all_headways = pd.read_parquet("data/stib_all_headways.parquet")
# remove line when the line arrive to destination
#all_headways = all_headways[all_headways['trip_headsign'] != #all_headways['stop_name']]
all_headways


In [ ]:
all_headways[all_headways.headway_minutes > 120]


In [ ]:
all_headways.loc[all_headways.headway_minutes > 120, "headway_minutes"] = float("nan")


In [ ]:
all_headways[(all_headways.route_short_name == '5')] #['stop_name'].value_counts()


In [ ]:
all_headways[(all_headways['trip_headsign'] == all_headways['stop_name']) & (all_headways.route_short_name == '5') & (all_headways.direction_id == 0)]


In [ ]:
all_headways[(all_headways['trip_headsign'] == all_headways['stop_name']) & (all_headways.route_short_name == '5')]


In [ ]:
all_headways[(all_headways['trip_headsign'] == all_headways['stop_name']) & (all_headways.route_short_name == '5')]


### Avg headways hour


In [ ]:
mean_headways = (
    all_headways
    .groupby(['date','route_short_name' ,'direction_id','stop_id',  'stop_name','arrival_hour'], sort=False)['headway_minutes']
    .mean()
    .reset_index()
    .rename(columns={'headway_minutes': 'mean_headway'})
)
mean_headways


In [ ]:
headways_clean = all_headways.merge(
    mean_headways,
    on=['date','route_short_name' ,'direction_id', 'stop_id', 'stop_name',    'arrival_hour'],
    how='left',
    sort=False
).reset_index(drop=True)
headways_clean = headways_clean[['stop_sequence', 'stop_id', 'stop_name',  'service_id', 'trip_id', 'headway_minutes', 'date','trip_headsign', 'direction_id', 'route_short_name',   'arrival_time_obj' ,'arrival_hour', 'mean_headway']]
headways_clean


In [ ]:
headways_clean[(headways_clean['trip_headsign'] == headways_clean['stop_name']) & (headways_clean.route_short_name == '5') & (headways_clean.direction_id == 0)]


##Time group pattern By line stops


In [ ]:
import ruptures as rpt


### Ruptures detection auto:
```python
model = rpt.Pelt(model="l2").fit(df_sub["headway"].values)
breaks = model.predict(pen=3)
```


In [ ]:
def define_time_groups_auto(df, route_short_name, stop_name, date, direction_id=0, pen=3):
    """
    Détecte automatiquement les time-groups (TG) à partir des headways
    d'une ligne, direction, arrêt et date donnés.

    Parameters
    ----------
    df : pd.DataFrame
        Doit contenir les colonnes ['date', 'arrival_hour', 'headway', 'stop_name', 'route_short_name', 'direction_id']
    line_id : int
        Identifiant de la ligne (ex: 71)
    stop_name : str
        Nom de l'arrêt (ex: 'ULB')
    date : str
        Date au format 'YYYY-MM-DD'
    direction_id : int, default=0
        Sens de circulation (0 ou 1)
    pen : float, default=3
        Paramètre de pénalité pour la détection de ruptures (plus grand = moins de segments)

    Returns
    -------
    df_out : pd.DataFrame
        DataFrame original enrichi d'une colonne 'tg_id'
    tg_summary : pd.DataFrame
        Résumé des time-groups (plages horaires et headway moyen)
    """
    date = pd.to_datetime(date)

    # Filtrage du sous-ensemble
    df_sub = df[
        (df["date"] == date) &
        (df["stop_name"] == stop_name) &
        (df["route_short_name"] == route_short_name) &
        (df["direction_id"] == direction_id)
    ].sort_values("arrival_hour").reset_index(drop=True)

    if df_sub.empty:
        raise ValueError("Aucune donnée pour ce filtre.")

    # Série à segmenter
    signal = df_sub["mean_headway"].values

    # Application du modèle de détection de ruptures
    model = rpt.Pelt(model="l2").fit(signal)
    breakpoints = model.predict(pen=pen)

    # Convertir les bornes en TG
    df_sub["tg_id"] = 0
    last_bkp = 0
    for i, bkp in enumerate(breakpoints):
        df_sub.loc[last_bkp:bkp-1, "tg_id"] = i
        last_bkp = bkp

    # Résumé des TG
    tg_summary = (
        df_sub.groupby("tg_id")
        .agg(
            start_hour=("arrival_hour", "min"),
            end_hour=("arrival_hour", "max"),
            mean_headway=("mean_headway", "mean")
        )
        .reset_index()
    )

    return df_sub, tg_summary


In [ ]:
mean_headways['date'] = pd.to_datetime(mean_headways['date'])
mean_headways


In [ ]:
mean_headways[(mean_headways.stop_name == 'ULB') & (mean_headways.route_short_name == '71')]


In [ ]:
df_tg_a, summary_a = define_time_groups_auto(
    mean_headways,
    route_short_name='71',
    stop_name="ULB",
    date="2025-09-08",
    direction_id=0,
    pen=3
)


In [ ]:
summary_a


In [ ]:
df_tg_a


In [ ]:
for pen in [2, 3, 5, 8]:
    df_tg_, summary_ = define_time_groups_auto(mean_headways, '71', "ULB", "2025-09-08", 0, pen)
    print(f"pen={pen} ➜ {len(summary_)} groupes détectés")


In [ ]:
plt.figure(figsize=(10,5))
plt.plot(df_tg_a["arrival_hour"], df_tg_a["mean_headway"], label="Headway (min)", marker='o')
for _, row in summary_a.iterrows():
    plt.axvspan(row["start_hour"], row["end_hour"], color='red', alpha=0.2)
plt.xlabel("Hour")
plt.xticks(range(df_tg_a["arrival_hour"].min(), df_tg_a["arrival_hour"].max() + 1, 1))
plt.ylabel("Headway (minutes)")
plt.title("Detected Time Groups — line 71, stop ULB (2025-09-08)")
plt.legend()
plt.show()


### Ruptures detection manuel:


In [ ]:
def define_time_groups(df, route_short_name, stop_name, date,
                       direction_id=0, threshold=2.5):
    """
    Crée des time groups selon la stabilité des headways.
    """
    df = df.copy().reset_index(drop=True)
    date = pd.to_datetime(date) 

    df = df[
        (df["date"] == date) &
        (df["stop_name"] == stop_name) &
        (df["route_short_name"] == route_short_name) &
        (df["direction_id"] == direction_id)
        ].sort_values("arrival_hour").reset_index(drop=True)

    # rolling
    #df["headway_smooth"] = df["headway"].rolling(window=3, center=True, min_periods=1).mean()

    #df["headway_smooth"] = df["headway"].rolling(window=3, center=True, min_periods=1).mean()
    df["diff"] = df["mean_headway"].diff().abs()
    df["new_group"] = (df["diff"] > threshold).astype(int)
    df["tg_id"] = df["new_group"].cumsum()  # incrémente à chaque rupture
    df["day_name"] = (df["date"]).dt.day_name()
    summary = (
        df.groupby("tg_id")
        .agg(start_hour=("arrival_hour", "min"),
             end_hour=("arrival_hour", "max"),
             mean_headway=("mean_headway", "mean"))
        .reset_index()
    )
    summary["date"] = date
    summary["day_name"] = (df["date"]).dt.day_name()
    return df.drop(columns=["new_group", "diff"]) , summary


In [ ]:
df_tg_m, summary_m = define_time_groups(
    mean_headways,
    route_short_name='71',
    stop_name="ULB",
    date="2025-09-08",
    direction_id=0,
    threshold=1.25
)


In [ ]:
df_tg_m


In [ ]:
summary_m


In [ ]:
def _():
    plt.figure(figsize=(10,5))
    plt.plot(df_tg_m["arrival_hour"], df_tg_m["mean_headway"], label="Headway (min)", marker='o')
    for _, row in summary_m.iterrows():
        plt.axvspan(row["start_hour"], row["end_hour"], color='red', alpha=0.2)
    plt.xlabel("Hour")
    plt.xticks(range(df_tg_m["arrival_hour"].min(), df_tg_m["arrival_hour"].max() + 1, 1))
    plt.ylabel("Headway (minutes)")
    plt.title("Detected Time Groups — line 71, stop ULB (2025-09-08)")
    plt.legend()
    return plt.show()


_()


In [ ]:
def _():
    plt.figure(figsize=(10,5))
    plt.plot(df_tg_m1["arrival_hour"], df_tg_m1["mean_headway"], label="Headway (min)", marker='o')
    for _, row in summary_m1.iterrows():
        plt.axvspan(row["start_hour"], row["end_hour"], color='red', alpha=0.2)
    plt.xlabel("Hour")
    plt.xticks(range(df_tg_m1["arrival_hour"].min(), df_tg_m1["arrival_hour"].max() + 1, 1))
    plt.ylabel("Headway (minutes)")
    plt.title("Detected Time Groups — line 71, stop TRONE (2025-09-08)")
    plt.legend()
    return plt.show()


_()


In [ ]:
mean_headways[ (mean_headways.route_short_name == '71')]['stop_name'].unique()


In [ ]:
df_tg_m1, summary_m1 = define_time_groups(
    mean_headways,
    route_short_name='71',
    stop_name="TRONE",
    date="2025-09-08",
    direction_id=0,
    threshold=1.25
)


In [ ]:
summary_m


In [ ]:
summary_m1


In [ ]:
df_tg_m1


### time group day-pattern line:
bus 71 stop ULB


In [ ]:
mean_headways


In [ ]:
from tqdm import tqdm


In [ ]:
all_tg = []
all_summary = []

for date in tqdm(mean_headways["date"].unique()):
    try:
        df_tg , df_summary = define_time_groups(
            mean_headways,
            route_short_name='71',
            stop_name="ULB",
            date=date,
            direction_id=0,
            threshold=1.25
        )
        all_tg.append(df_tg)
        all_summary.append(df_summary)
    except Exception as e:
        print(f"⚠️ {date}: {e}")

all_tg = pd.concat(all_tg).reset_index(drop=True)
all_summary = pd.concat(all_summary).reset_index(drop=True)


In [ ]:
all_tg


In [ ]:
all_summary


In [ ]:
patterns = (
    all_summary.groupby(["day_name", "tg_id"])
    .agg(mean_start=("start_hour", "mean"),
         mean_end=("end_hour", "mean"),
         mean_headway=("mean_headway", "mean"),
         n_days=("date", "nunique"))
    .reset_index()
)
patterns


In [ ]:
df_pivot = all_tg.pivot_table(
    index="date",
    columns="arrival_hour",
    values="mean_headway",
    aggfunc="mean"
)#.fillna(method="ffill", axis=1)

df_pivot.columns = [f"h{col}" for col in df_pivot.columns]

df_pivot = df_pivot.loc[ : ,  :'h24']
df_pivot


In [ ]:
df_plot = df_pivot.copy()
df_plot.index = df_plot.index.strftime('%Y-%m-%d')  # <-- Enlève les heures


In [ ]:
plt.figure(figsize=(12,5))
sns.heatmap(df_plot, cmap="YlOrRd", cbar_kws={'label': 'Headway (min)'})
plt.title("Headway patterns per day (Line 71, stop ULB)")
plt.xlabel("Hour of day")
plt.ylabel("Date")
plt.show()


In [ ]:
def _():
    # Créer la heatmap Plotly
    fig = px.imshow(
        df_plot,
        color_continuous_scale="YlOrRd",
        labels=dict(x="Hour of day", y="Date", color="Headway (min)"),
        title="Headway patterns per day (Line 71, stop ULB)"
    )

    # Ajustements esthétiques
    fig.update_layout(
        width=750,
        height=450,
        xaxis_nticks=20,
        font=dict(size=12, color="black"),
        title_x=0.5
    )
    return fig.show()


_()


### Time group Pattern kmeans


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans


In [ ]:
X = df_pivot.values

# Normalisation (par heure)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Clustering (3 groupes typiques : ouvrables, samedi, dimanche)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_scaled)

# Ajouter les résultats au df_pivot
df_pivot_clustered = df_pivot.copy()
df_pivot_clustered["cluster"] = labels

# Ajouter le jour de la semaine
df_pivot_clustered["day_name"] = pd.to_datetime(df_pivot_clustered.index).day_name()


In [ ]:
df_pivot_clustered


In [ ]:
cluster_summary = (
    df_pivot_clustered.groupby("cluster")["day_name"]
    .agg(["count", lambda x: list(x.unique())])
    .rename(columns={"<lambda_0>": "days_in_cluster"})
).reset_index()
cluster_summary


In [ ]:
plt.figure(figsize=(10,6))
for cluster_id in sorted(df_pivot_clustered["cluster"].unique()):
    profile = df_pivot_clustered[df_pivot_clustered["cluster"] == cluster_id].iloc[:, :-2].mean()
    plt.plot(profile.index, profile.values, marker='o', label=f"Cluster {cluster_id}")

plt.xlabel("Hour of day")
plt.ylabel("Headway (min)")
plt.title("Average daily headway profiles by cluster")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
df_pivot_clustered[df_pivot_clustered.cluster == 2]


#### Statistiques globales (par jour)


In [ ]:
stats_day = pd.DataFrame({
    "mean_headway": df_pivot.mean(axis=1),
    "std_headway": df_pivot.std(axis=1),
    "min_headway": df_pivot.min(axis=1),
    "max_headway": df_pivot.max(axis=1),
})
# coefficient de variation cv_headway
stats_day["cv_headway"] = stats_day["std_headway"] / stats_day["mean_headway"]
stats_day["day_name"] = pd.to_datetime(stats_day.index).day_name()

stats_day


#### Agrégation par jour de la semaine


In [ ]:
weekly_pattern = (
    stats_day.groupby("day_name")
    .agg({
        "mean_headway": ["mean", "std"],
        "cv_headway": ["mean", "std"],
    })
    .sort_values(("mean_headway", "mean"))
)

weekly_pattern


#### Moyenne du headway par jour de la semaine


In [ ]:
plt.figure(figsize=(8,4))
sns.barplot(data=stats_day, x="day_name", y="mean_headway", order=[
    "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"])
plt.ylabel("Mean headway (min)")
plt.title("Average daily headway by weekday (Line 71, stop ULB)")
plt.show()


#### Coefficient de variation


In [ ]:
plt.figure(figsize=(8,4))
sns.barplot(data=stats_day, x="day_name", y="cv_headway",
            order=["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"])
plt.ylabel("Headway variability (CV)")
plt.title("Headway regularity by weekday (lower = more regular)")
plt.show()


## Time group pattern By line


### Bus 71


In [ ]:
mean_headways.columns


In [ ]:
mean_headways


In [ ]:
df_pivot_71_0 = build_line_pivot(mean_headways, route_short_name='71', direction_id=0)
#df_pivot_71_0.isnull().sum()
df_pivot_71_0


In [ ]:
df_pivot_71_0_plot = df_pivot_71_0.copy()
# <-- Enlève les heures
df_pivot_71_0_plot.index = df_pivot_71_0_plot.index #.strftime('%Y-%m-%d')


In [ ]:
def _():
    trip_headsign = timetable[(timetable.route_short_name == '71') & (timetable.direction_id == 0) ]['trip_headsign'].unique()[0]

    plt.figure(figsize=(12,5))
    sns.heatmap(df_pivot_71_0_plot, cmap="YlOrRd", cbar_kws={'label': 'Headway (min)'})
    plt.title(f"Headway patterns per day — Line 71, direction {trip_headsign} ")
    plt.xlabel("Hour of day")
    plt.ylabel("Date")
    return plt.show()
_()


In [ ]:
def _():
    trip_headsign = timetable[(timetable.route_short_name == '71') & (timetable.direction_id == 0) ]['trip_headsign'].unique()[0]

    # Créer la heatmap Plotly
    fig = px.imshow(
        df_pivot_71_0_plot,
        color_continuous_scale="YlOrRd",
        labels=dict(x="Hour of day", y="Date", color="Headway (min)"),
        title=f"Headway patterns per day (Line 71, direction {trip_headsign})"
    )

    # Ajustements esthétiques
    fig.update_layout(
        width=750,
        height=450,
        xaxis_nticks=20,
        font=dict(size=12, color="black"),
        title_x=0.5
    )
    return fig.show()


_()


### metro 5


In [ ]:
df_pivot_5_0 = build_line_pivot(headways_clean, route_short_name='5', direction_id=0)
df_pivot_5_0


In [ ]:
df_pivot_5_0_plot = df_pivot_5_0.copy()
# <-- Enlève les heures
df_pivot_5_0_plot.index = df_pivot_5_0_plot.index #.strftime('%Y-%m-%d')


In [ ]:
trip_headsign = timetable[(timetable.route_short_name == '5') & (timetable.direction_id == 0) ]['trip_headsign'].unique()[0]

plt.figure(figsize=(12,5))
sns.heatmap(df_pivot_5_0_plot, cmap="YlOrRd", cbar_kws={'label': 'Headway (min)'})
plt.title(f"Headway patterns per day — Line 5, direction {trip_headsign}")
plt.xlabel("Hour of day")
plt.ylabel("Date")
plt.show()


In [ ]:
def _():

    trip_headsign = timetable[(timetable.route_short_name == '5') & (timetable.direction_id == 0) ]['trip_headsign'].unique()[0]
    # Créer la heatmap Plotly
    fig = px.imshow(
        df_pivot_5_0_plot,
        color_continuous_scale="YlOrRd",
        labels=dict(x="Hour of day", y="Date", color="Headway (min)"),
        title=f"Headway patterns per day (Line 5, direction : {trip_headsign})"
    )

    # Ajustements esthétiques
    fig.update_layout(
        width=750,
        height=450,
        xaxis_nticks=20,
        font=dict(size=12, color="black"),
        title_x=0.5
    )
    return fig.show()


_()


### metro 6


In [ ]:
df_pivot_6_0 = build_line_pivot(mean_headways, route_short_name='6', direction_id=0)
df_pivot_6_0


In [ ]:
df_pivot_6_0_plot = df_pivot_6_0.copy()
# <-- Enlève les heures
df_pivot_6_0_plot.index = df_pivot_6_0_plot.index #.strftime('%Y-%m-%d')


In [ ]:
def _():
    trip_headsign = timetable[(timetable.route_short_name == '6') & (timetable.direction_id == 0) ]['trip_headsign'].unique()[0]

    plt.figure(figsize=(12,5))
    sns.heatmap(df_pivot_6_0_plot, cmap="YlOrRd", cbar_kws={'label': 'Headway (min)'})
    plt.title(f"Headway patterns per day — Line 6, direction {trip_headsign}")
    plt.xlabel("Hour of day")
    plt.ylabel("Date")
    return plt.show()
_()


In [ ]:
def _():

    trip_headsign = timetable[(timetable.route_short_name == '6') & (timetable.direction_id == 0) ]['trip_headsign'].unique()[0]
    # Créer la heatmap Plotly
    fig = px.imshow(
        df_pivot_6_0_plot,
        color_continuous_scale="YlOrRd",
        labels=dict(x="Hour of day", y="Date", color="Headway (min)"),
        title=f"Headway patterns per day (Line 6, direction : {trip_headsign})"
    )

    # Ajustements esthétiques
    fig.update_layout(
        width=750,
        height=450,
        xaxis_nticks=20,
        font=dict(size=12, color="black"),
        title_x=0.5
    )
    return fig.show()
_()


## Time group all


In [ ]:
def map_hour_to_timegroup(hour):
        """
        Map a single hour to its timegroup.

        Parameters:
        -----------
        hour : int
            Hour of day (0-23)

        Returns:
        --------
        str
            Timegroup label
        """
        if pd.isna(hour):
            return "Unknown"

        hour = int(hour)

        if 5 <= hour <= 6:
            return "Early Morning"
        elif 7 <= hour <= 9:
            return "Morning Peak"
        elif 10 <= hour <= 15:
            return "Off-Peak Day"
        elif 16 <= hour <= 19:
            return "Evening Peak"
        elif 20 <= hour <= 22:
            return "Late Night"
        else:
            return "Night"


In [ ]:
headways_clean['time_group'] = headways_clean['arrival_hour'].apply(map_hour_to_timegroup)


## Distinguish type assessment


In [ ]:
def classify_qos_assessment(headway_minutes):
    """
    Classify QoS assessment type based on headway.

    Parameters:
    -----------
    headway_minutes : float
        Headway in minutes

    Returns:
    --------
    str
        'Regularity' if headway < 12 min, 'Punctuality' otherwise
    """
    if pd.isna(headway_minutes):
        return 'Unknown'
    elif headway_minutes < 12:
        return 'Regularity'
    else:
        return 'Punctuality'


In [ ]:
headways_clean["qos_assessment"] = headways_clean['mean_headway'].apply(classify_qos_assessment)
# regularity
df_regularity = headways_clean[headways_clean["qos_assessment"] == 'Regularity']
# punctuality
df_punctuality = headways_clean[headways_clean["qos_assessment"] == 'Punctuality']


In [ ]:
df_regularity.columns


In [ ]:
df_regularity


In [ ]:
df_regularity[(df_regularity['trip_headsign'] == df_regularity['stop_name']) & (df_regularity.route_short_name == '5') & (df_regularity.direction_id == 0)]


In [ ]:
df_punctuality


In [ ]:
df_punctuality[(df_punctuality['trip_headsign'] == df_punctuality['stop_name']) & (df_punctuality.route_short_name == '6') & (df_punctuality.direction_id == 0)]


In [ ]:
headways_clean.to_parquet("data/all_clean_headways.parquet" , index=False)


In [ ]:
df_regularity.to_parquet("data/stib_all_regularity.parquet", index=False)
df_punctuality.to_parquet("data/stib_all_punctuality.parquet", index=False)


In [ ]:
df_punctuality.columns


In [ ]:
df_punctuality[(df_punctuality.route_short_name == '5') & (df_punctuality.direction_id == 0)]['stop_name'].unique()


In [ ]:
df_regularity[(df_regularity.stop_name == 'PASTEUR') & (df_regularity.route_short_name == '37') & (df_regularity.direction_id == 0) ]['arrival_hour'].unique()
